# 📍 Minería de Datos — KNN: K-Nearest Neighbors
## Clase comparación con Árboles de Decisión + taller final

**Asignatura:** Minería de Datos  
**Tema:** Clasificación supervisada con KNN  
**Duración estimada:** 2 horas  
**Nivel:** Ingeniería de Sistemas  

---

## Propósito de la clase

En la clase anterior se estudió un modelo basado en reglas: **Árboles de Decisión**. En esta clase estudiaremos un modelo totalmente diferente: **KNN — K-Nearest Neighbors**.

La idea central de KNN es sencilla:

> Para clasificar un nuevo caso, busque los casos más parecidos en los datos históricos y tome una decisión por votación.

Usaremos nuevamente el dataset **Titanic** para mantener continuidad con la clase anterior.

---

## Resultados esperados de aprendizaje

Al finalizar la clase, el estudiante debe poder:

1. Explicar la intuición de KNN.
2. Entender el concepto de vecino más cercano.
3. Calcular e interpretar distancia euclidiana.
4. Explicar por qué KNN depende fuertemente de la escala.
5. Entrenar un clasificador KNN en Python.
6. Comparar KNN sin escalado y con escalado.
7. Evaluar KNN usando métricas de clasificación.
8. Analizar el efecto de elegir diferentes valores de \(k\).
9. Comparar KNN contra árbol de decisión.
10. Justificar qué modelo usaría según el contexto.

# 1. Ubicación dentro del curso

Hasta este punto se han trabajado conceptos generales de minería de datos, CRISP-DM, preparación de datos, train/test, métricas de clasificación, regresión logística y árboles de decisión.

Ahora seguimos consolidando clasificación supervisada con un modelo basado en distancia.

| Modelo | ¿Cómo toma decisiones? | Tipo de lógica |
|---|---|---|
| Regresión logística | Probabilidad mediante una función | Modelo paramétrico |
| Árbol de decisión | Reglas IF-THEN | Modelo basado en reglas |
| KNN | Vecinos más parecidos | Modelo basado en similitud |

La clase anterior el modelo preguntaba:

\`\`\`text
¿Sex <= 0.5?
¿Pclass <= 2?
¿Age <= 12?
\`\`\`

Hoy el modelo pregunta:

\`\`\`text
¿A qué pasajeros históricos se parece más este nuevo pasajero?
\`\`\`

# 2. Intuición de KNN

KNN significa **K-Nearest Neighbors**, o **K vecinos más cercanos**.

La lógica es:

1. Tengo un nuevo caso sin clasificar.
2. Calculo la distancia entre ese caso y todos los casos del entrenamiento.
3. Selecciono los \(k\) casos más cercanos.
4. Observo sus clases.
5. Asigno la clase más frecuente.

Ejemplo con \(k=5\):

\`\`\`text
Vecino 1 → sobrevivió
Vecino 2 → sobrevivió
Vecino 3 → no sobrevivió
Vecino 4 → sobrevivió
Vecino 5 → no sobrevivió
\`\`\`

Votación: sobrevivió 3 votos, no sobrevivió 2 votos. Predicción: sobrevivió.

KNN no construye reglas explícitas como un árbol. KNN guarda los datos y usa similitud al momento de predecir.

# 3. ¿KNN realmente “entrena”?

Técnicamente, KNN es un modelo de aprendizaje perezoso o **lazy learning**.

- Durante el entrenamiento, casi no aprende parámetros.
- Guarda los datos.
- El trabajo fuerte ocurre al momento de predecir.

| Modelo | Qué hace al entrenar | Qué hace al predecir |
|---|---|---|
| Árbol | Aprende reglas | Sigue las reglas |
| Regresión logística | Aprende coeficientes | Calcula probabilidad |
| KNN | Guarda datos | Busca vecinos y vota |

Implicaciones:

- Puede ser lento al predecir si hay muchos datos.
- Necesita una buena representación de las variables.
- Es muy sensible a la escala.

# 4. Distancia euclidiana

La distancia más común en KNN es la distancia euclidiana:

\[
d(x,z) = \sqrt{\sum_{j=1}^{p}(x_j - z_j)^2}
\]

Donde \(p\) es el número de variables. Si dos pasajeros tienen valores parecidos en las variables usadas, estarán cerca. Si son muy diferentes, estarán lejos.

En dos dimensiones:

\[
d(x,z) = \sqrt{(x_1-z_1)^2 + (x_2-z_2)^2}
\]

In [ ]:
# ============================================================
# 1. Importar librerías
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)

# 5. Ejemplo manual de distancia

Supongamos dos pasajeros representados por edad y tarifa pagada.

Pasajero A: \(A = (22, 7.25)\)  
Pasajero B: \(B = (38, 71.28)\)

\[
d(A,B)=\sqrt{(22-38)^2 + (7.25-71.28)^2}
\]

In [ ]:
pasajero_A = np.array([22, 7.25])
pasajero_B = np.array([38, 71.28])

distancia = np.sqrt(np.sum((pasajero_A - pasajero_B) ** 2))
print("Distancia entre A y B:", round(distancia, 2))

## Discusión

La variable [96mFare[0m puede dominar la distancia porque sus valores pueden ser mucho más grandes que variables como [96mPclass[0m, [96mSibSp[0m o [96mParch[0m.

Esto nos lleva al punto más importante de KNN:

# KNN necesita escalado.

# 6. Problema de escala

KNN calcula distancias. Si una variable tiene valores grandes, puede dominar la distancia.

| Variable | Rango aproximado |
|---|---|
| Pclass | 1 a 3 |
| Sex | 0 a 1 |
| Age | 0 a 80 |
| Fare | 0 a 500 |

La variable [96mFare[0m puede pesar mucho más que [96mSex[0m o [96mPclass[0m. Esto no significa que sea más importante, sino que tiene una escala mayor.

## Solución: estandarización

\[
z = \frac{x - \mu}{\sigma}
\]

Después de estandarizar, cada variable tiene media aproximada 0 y desviación estándar aproximada 1.

# 7. Dataset Titanic

Usaremos el mismo dataset de la clase de árboles de decisión.

URL:

\`\`\`python
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
\`\`\`

El objetivo es predecir \(Survived\):

- 0: No sobrevivió.
- 1: Sobrevivió.

In [ ]:
# ============================================================
# 2. Cargar dataset Titanic desde URL
# ============================================================

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)
df.head()

# 8. Comprensión inicial del dataset

Antes de modelar revisamos estructura, faltantes, distribución de la variable objetivo y seleccionamos variables para modelar.

In [ ]:
df.info()

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
}).sort_values("faltantes", ascending=False)

faltantes

In [ ]:
conteo = df["Survived"].value_counts().sort_index()
porcentaje = df["Survived"].value_counts(normalize=True).sort_index() * 100

resumen_target = pd.DataFrame({
    "Survived": conteo.index,
    "descripcion": ["No sobrevivió", "Sobrevivió"],
    "conteo": conteo.values,
    "porcentaje": porcentaje.round(2).values
})

resumen_target

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(resumen_target["descripcion"], resumen_target["conteo"])
plt.title("Distribución de supervivencia")
plt.xlabel("Clase")
plt.ylabel("Número de pasajeros")
plt.show()

# 9. Exploración rápida orientada a KNN

KNN usa distancias, entonces debemos mirar variables numéricas, rangos, escala y variables categóricas codificadas.

Usaremos variables simples:

| Variable | Tipo |
|---|---|
| Pclass | numérica ordinal |
| Sex | categórica transformada |
| Age | numérica |
| SibSp | numérica |
| Parch | numérica |
| Fare | numérica |
| Embarked | categórica transformada |

In [ ]:
tabla_sex = pd.crosstab(df["Sex"], df["Survived"], normalize="index") * 100
tabla_sex.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_sex.round(2)

In [ ]:
tabla_sex.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por sexo")
plt.xlabel("Sexo")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

In [ ]:
tabla_clase = pd.crosstab(df["Pclass"], df["Survived"], normalize="index") * 100
tabla_clase.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_clase.round(2)

In [ ]:
tabla_clase.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por clase del pasajero")
plt.xlabel("Clase")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

# 10. Preparación de datos

Para KNN debemos seleccionar variables útiles, manejar faltantes, convertir categóricas, separar X/y, separar train/test y escalar variables usando solo train.

## Cuidado con data leakage

El escalador debe aprender la media y desviación estándar usando solo entrenamiento. Por eso usaremos [96mPipeline[0m.

El pipeline garantiza:

1. Escala usando train.
2. Entrena KNN con train escalado.
3. Al predecir test, usa el escalado aprendido desde train.

In [ ]:
df_model = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

# Imputación de valores faltantes
df_model["Age"] = df_model["Age"].fillna(df_model["Age"].median())
df_model["Embarked"] = df_model["Embarked"].fillna(df_model["Embarked"].mode()[0])

# Codificación de Sex
df_model["Sex"] = df_model["Sex"].map({"male": 0, "female": 1})

# One-hot encoding de Embarked
df_model = pd.get_dummies(df_model, columns=["Embarked"], drop_first=True)

print("Dimensiones del dataset modelado:", df_model.shape)
df_model.head()

In [ ]:
df_model.isna().sum()

In [ ]:
X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]

print("X:", X.shape)
print("y:", y.shape)
print("Variables predictoras:", list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("\nDistribución train:")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribución test:")
print(y_test.value_counts(normalize=True).sort_index())

# 11. KNN sin escalado: demostración del problema

Primero entrenaremos KNN sin escalar. Esto no es lo recomendado, pero sirve para demostrar por qué la escala importa.

Usaremos \(k=5\). El parámetro \(k\) indica cuántos vecinos se usan para votar.

In [ ]:
knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
knn_sin_escalar.fit(X_train, y_train)
pred_sin_escalar = knn_sin_escalar.predict(X_test)

print("Resultados KNN sin escalado")
print(classification_report(y_test, pred_sin_escalar, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
cm = confusion_matrix(y_test, pred_sin_escalar)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No sobrevivió", "Sobrevivió"])
disp.plot()
plt.title("Matriz de confusión - KNN sin escalado")
plt.show()

## Discusión

Preguntas para estudiantes:

1. ¿Qué tan bueno parece el modelo?
2. ¿Qué variables pueden estar dominando la distancia?
3. ¿Por qué [96mFare[0m puede afectar mucho?
4. ¿Qué pasaría si agregáramos una variable como ingreso anual con valores en millones?

Conclusión esperada: KNN no debe usarse sin revisar escalas.

# 12. KNN con escalado usando Pipeline

Ahora usaremos el enfoque correcto. El pipeline tendrá dos pasos:

1. [96mStandardScaler()[0m: estandariza variables.
2. [96mKNeighborsClassifier()[0m: entrena KNN.

Esto evita fuga de información porque el escalado se ajusta solo con train.

In [ ]:
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

pipeline_knn.fit(X_train, y_train)
pred_knn = pipeline_knn.predict(X_test)

print("Resultados KNN con escalado")
print(classification_report(y_test, pred_knn, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
cm = confusion_matrix(y_test, pred_knn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No sobrevivió", "Sobrevivió"])
disp.plot()
plt.title("Matriz de confusión - KNN con escalado")
plt.show()

# 13. Comparación: KNN sin escalado vs KNN con escalado

Construiremos una tabla para comparar accuracy, precision, recall y F1-score.

In [ ]:
def evaluar_modelo(nombre, y_real, y_pred):
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred),
        "recall": recall_score(y_real, y_pred),
        "f1": f1_score(y_real, y_pred)
    }

comparacion_escalado = pd.DataFrame([
    evaluar_modelo("KNN sin escalado", y_test, pred_sin_escalar),
    evaluar_modelo("KNN con escalado", y_test, pred_knn)
])
comparacion_escalado

## Interpretación esperada

- KNN depende de distancias.
- Las distancias dependen de la escala.
- Por eso el escalado no es un detalle técnico menor: es parte del modelo.

> En KNN, preprocesar bien no es opcional; es parte de la definición del modelo.

# 14. ¿Cómo elegir \(k\)?

El parámetro \(k\) define cuántos vecinos votan.

## Si k es muy pequeño

- El modelo es sensible al ruido.
- Puede sobreajustar.
- Las fronteras de decisión son irregulares.

## Si k es muy grande

- El modelo se vuelve demasiado general.
- Puede subajustar.
- Puede favorecer la clase mayoritaria.

| k | Comportamiento |
|---|---|
| k = 1 | Muy flexible, sensible al ruido |
| k = 3 o 5 | Balance inicial frecuente |
| k grande | Más estable, pero puede perder detalle |

# 15. Evaluación de diferentes valores de k

Entrenaremos KNN con distintos valores de \(k\) y compararemos resultados. Usaremos pipeline para escalar siempre correctamente.

In [ ]:
valores_k = list(range(1, 31))
resultados_k = []

for k in valores_k:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    modelo.fit(X_train, y_train)
    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)
    resultados_k.append({
        "k": k,
        "accuracy_train": accuracy_score(y_train, pred_train),
        "accuracy_test": accuracy_score(y_test, pred_test),
        "f1_test": f1_score(y_test, pred_test),
        "recall_test": recall_score(y_test, pred_test),
        "precision_test": precision_score(y_test, pred_test)
    })

df_k = pd.DataFrame(resultados_k)
df_k.head()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df_k["k"], df_k["accuracy_train"], marker="o", label="Train")
plt.plot(df_k["k"], df_k["accuracy_test"], marker="o", label="Test")
plt.title("Efecto de k sobre Accuracy")
plt.xlabel("Número de vecinos k")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df_k["k"], df_k["f1_test"], marker="o", label="F1 test")
plt.plot(df_k["k"], df_k["recall_test"], marker="o", label="Recall test")
plt.plot(df_k["k"], df_k["precision_test"], marker="o", label="Precision test")
plt.title("Efecto de k sobre métricas de clasificación")
plt.xlabel("Número de vecinos k")
plt.ylabel("Valor de métrica")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
df_k.sort_values("f1_test", ascending=False).head(10)

## Interpretación

Preguntas para clase:

1. ¿Qué valor de k da mejor F1?
2. ¿Qué pasa con k = 1?
3. ¿Qué pasa cuando k crece mucho?
4. ¿Hay diferencia entre accuracy y F1?
5. ¿Elegiría k solo por la métrica o también por estabilidad?

En proyectos reales se usa validación cruzada. En esta clase usamos train/test para mantener el foco conceptual.

# 16. Entrenar KNN final con el mejor k observado

Seleccionaremos el mejor k según F1-score en test para efectos pedagógicos. En un proyecto real, este ajuste debería hacerse con validación cruzada sobre entrenamiento, no usando test repetidamente.

In [ ]:
mejor_k = int(df_k.sort_values("f1_test", ascending=False).iloc[0]["k"])
print("Mejor k observado según F1 test:", mejor_k)

knn_final = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=mejor_k))
])

knn_final.fit(X_train, y_train)
pred_knn_final = knn_final.predict(X_test)

print(classification_report(y_test, pred_knn_final, target_names=["No sobrevivió", "Sobrevivió"]))

# 17. Comparación contra Árbol de Decisión

Ahora compararemos KNN con el árbol visto en la clase anterior.

| Modelo | Lógica |
|---|---|
| Árbol | Reglas |
| KNN | Similitud |

In [ ]:
arbol = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
arbol.fit(X_train, y_train)
pred_arbol = arbol.predict(X_test)

comparacion_modelos = pd.DataFrame([
    evaluar_modelo("Árbol de decisión", y_test, pred_arbol),
    evaluar_modelo(f"KNN final k={mejor_k}", y_test, pred_knn_final)
])
comparacion_modelos

In [ ]:
metricas = ["accuracy", "precision", "recall", "f1"]
comparacion_modelos.set_index("modelo")[metricas].plot(kind="bar", figsize=(10,5))
plt.title("Comparación Árbol vs KNN")
plt.ylabel("Valor")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.grid(axis="y")
plt.show()

## Discusión comparativa

1. ¿Cuál modelo tiene mejor accuracy?
2. ¿Cuál tiene mejor recall?
3. ¿Cuál tiene mejor F1?
4. ¿Cuál es más interpretable?
5. ¿Cuál depende más del preprocesamiento?
6. ¿Cuál sería más fácil de explicar a un usuario no técnico?
7. ¿Cuál podría escalar peor con muchísimos datos?

Lectura esperada: el árbol es más interpretable; KNN depende mucho del escalado; no existe un único mejor modelo, depende del objetivo.

# 18. Visualización sencilla en 2D

Para entender KNN de manera visual, usaremos solo dos variables: Age y Fare. Esto no será el mejor modelo, pero sirve para visualizar cómo KNN piensa en términos de cercanía.

In [ ]:
X_2d = df_model[["Age", "Fare"]]
y_2d = df_model["Survived"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_2d, y_2d, test_size=0.30, random_state=42, stratify=y_2d
)

knn_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
knn_2d.fit(X2_train, y2_train)

plt.figure(figsize=(8,5))
plt.scatter(X_2d["Age"], X_2d["Fare"], c=y_2d, alpha=0.7)
plt.title("Titanic: Age vs Fare coloreado por supervivencia")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.show()

## Interpretación

Cada punto es un pasajero. KNN clasifica un nuevo punto mirando qué puntos cercanos tiene alrededor. Con solo dos variables se pierde información, así que este gráfico es pedagógico, no necesariamente el mejor modelo.

# 19. Ventajas y desventajas de KNN

## Ventajas

- Intuitivo.
- Fácil de implementar.
- No asume una forma funcional específica.
- Puede capturar fronteras de decisión no lineales.
- Útil como modelo base.

## Desventajas

- Muy sensible a la escala.
- Puede ser costoso al predecir con muchos datos.
- Se afecta por variables irrelevantes.
- Se afecta por datos ruidosos.
- No genera reglas tan claras como un árbol.
- Elegir k requiere experimentación.

## Recomendación práctica

KNN se debe usar con buen preprocesamiento, escalado, revisión de variables, comparación contra otros modelos y validación adecuada.

# 20. Errores comunes al usar KNN

| Error | Consecuencia |
|---|---|
| No escalar variables | Distancias sesgadas |
| Usar variables irrelevantes | Vecinos mal definidos |
| Elegir k sin evaluar | Modelo inestable |
| Evaluar solo con accuracy | Interpretación incompleta |
| No separar train/test | Evaluación engañosa |
| Escalar antes del split | Data leakage |

> En KNN, la calidad de la distancia define la calidad del modelo.

# 🧪 21. Taller final de clase

## Objetivo

Entrenar, evaluar y comparar un modelo KNN usando Titanic.

## Entregable

Un notebook o documento con:

1. Preparación de datos.
2. KNN sin escalado.
3. KNN con escalado.
4. Comparación de valores de k.
5. Comparación contra árbol de decisión.
6. Matriz de confusión.
7. Conclusión profesional.

No se evalúa solo que el código funcione. Se evalúa principalmente la interpretación.

## Parte A — Comprensión conceptual

Responda:

1. ¿Qué significa KNN?
2. ¿Cómo clasifica KNN un nuevo registro?
3. ¿Qué representa el parámetro \(k\)?
4. ¿Por qué KNN depende de la distancia?
5. ¿Por qué el escalado es obligatorio o muy recomendable?
6. ¿Qué diferencia conceptual hay entre KNN y árbol de decisión?

## Parte B — Preparación de datos

Use las variables Pclass, Sex, Age, SibSp, Parch, Fare y Embarked. Realice imputación, codificación, separación X/y y train/test split con stratify. Explique por qué se hace cada paso.

In [ ]:
df_taller = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

df_taller["Age"] = df_taller["Age"].fillna(df_taller["Age"].median())
df_taller["Embarked"] = df_taller["Embarked"].fillna(df_taller["Embarked"].mode()[0])
df_taller["Sex"] = df_taller["Sex"].map({"male": 0, "female": 1})
df_taller = pd.get_dummies(df_taller, columns=["Embarked"], drop_first=True)

X_taller = df_taller.drop(columns=["Survived"])
y_taller = df_taller["Survived"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_taller, y_taller, test_size=0.30, random_state=42, stratify=y_taller
)
print(X_train_t.shape, X_test_t.shape)

## Parte C — KNN sin escalado vs KNN con escalado

Entrene KNN sin escalado y con escalado usando \(k=5\). Compare accuracy, precision, recall y F1-score. Responda cuál versión es metodológicamente más correcta.

In [ ]:
knn_sin = KNeighborsClassifier(n_neighbors=5)
knn_sin.fit(X_train_t, y_train_t)
pred_sin = knn_sin.predict(X_test_t)

knn_con = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
knn_con.fit(X_train_t, y_train_t)
pred_con = knn_con.predict(X_test_t)

comparacion_taller = pd.DataFrame([
    evaluar_modelo("KNN sin escalado", y_test_t, pred_sin),
    evaluar_modelo("KNN con escalado", y_test_t, pred_con)
])
comparacion_taller

## Parte D — Selección de k

Pruebe valores de \(k\) entre 1 y 30. Construya una tabla con accuracy, precision, recall y F1. Explique qué problema puede tener \(k=1\) y qué problema puede tener un \(k\) demasiado grande.

In [ ]:
resultados_k_taller = []
for k in range(1, 31):
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    modelo.fit(X_train_t, y_train_t)
    pred = modelo.predict(X_test_t)
    resultados_k_taller.append({
        "k": k,
        "accuracy_test": accuracy_score(y_test_t, pred),
        "precision_test": precision_score(y_test_t, pred),
        "recall_test": recall_score(y_test_t, pred),
        "f1_test": f1_score(y_test_t, pred)
    })

df_k_taller = pd.DataFrame(resultados_k_taller)
df_k_taller.sort_values("f1_test", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df_k_taller["k"], df_k_taller["f1_test"], marker="o", label="F1")
plt.plot(df_k_taller["k"], df_k_taller["recall_test"], marker="o", label="Recall")
plt.plot(df_k_taller["k"], df_k_taller["precision_test"], marker="o", label="Precision")
plt.title("Métricas según k - Taller")
plt.xlabel("k")
plt.ylabel("Métrica")
plt.legend()
plt.grid(True)
plt.show()

## Parte E — Comparación contra árbol

Entrene un árbol con [96mcriterion="gini"[0m y [96mmax_depth=3[0m. Compare contra el mejor KNN encontrado. Responda cuál modelo es más interpretable y cuál recomendaría según el objetivo.

In [ ]:
mejor_k_taller = int(df_k_taller.sort_values("f1_test", ascending=False).iloc[0]["k"])

knn_mejor_taller = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=mejor_k_taller))
])
knn_mejor_taller.fit(X_train_t, y_train_t)
pred_knn_mejor = knn_mejor_taller.predict(X_test_t)

arbol_taller = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
arbol_taller.fit(X_train_t, y_train_t)
pred_arbol_taller = arbol_taller.predict(X_test_t)

pd.DataFrame([
    evaluar_modelo(f"KNN k={mejor_k_taller}", y_test_t, pred_knn_mejor),
    evaluar_modelo("Árbol max_depth=3", y_test_t, pred_arbol_taller)
])

## Parte F — Conclusión profesional

Redacte una conclusión de mínimo 10 líneas respondiendo:

1. ¿Qué aprendió del comportamiento de KNN?
2. ¿Qué efecto tuvo el escalado?
3. ¿Qué valor de k eligió y por qué?
4. ¿KNN fue mejor o peor que el árbol?
5. ¿Qué métrica considera más relevante?
6. ¿Qué limitaciones tiene el análisis?
7. ¿Qué haría en una siguiente fase?

La conclusión debe sonar como un informe técnico dirigido a un equipo de datos.

# 22. Rúbrica sugerida del taller

| Criterio | Puntaje |
|---|---:|
| Comprensión conceptual de KNN | 0.7 |
| Preparación correcta de datos | 0.8 |
| Comparación sin escalado vs con escalado | 0.8 |
| Evaluación con métricas completas | 0.8 |
| Análisis de diferentes valores de k | 1.0 |
| Comparación contra árbol de decisión | 0.6 |
| Conclusión profesional | 0.3 |
| **Total** | **5.0** |

---

# Cierre de clase

KNN permite entender una idea fundamental en minería de datos:

> La forma en que representamos los datos cambia la forma en que el modelo aprende.

Con esta clase los estudiantes ya pueden comparar dos enfoques de clasificación:

| Modelo | Enfoque |
|---|---|
| Árbol | Reglas |
| KNN | Distancias |

La siguiente clase recomendada es **Naive Bayes**, porque introduce clasificación probabilística y permite contrastar reglas, distancias y probabilidad.